In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier #Multi layer Perceptron

## Analisi dataset di pacchetti firewall

 * 1 pulizia e lettura dei dati per capire bene con cosa ho a che fare

### Funzioni per grafici

Di seguito ricopio le funzioni per i grafici

In [52]:
def disegnaGrafico(x_name, y_name, data, cat_name, ax):
    lista = data[cat_name].unique()
    if x_name == y_name:
        for elemento in lista:
            selected_data = data[data[cat_name]== elemento]
            ax.hist(selected_data[x_name], alpha = 0.5, bins = 15, label=elemento)
            ax.set_xlabel(x_name)
            ax.set_ylabel("Numero di casi")
    else:
        for elemento in lista:
            selected_data = data[data[cat_name]== elemento]
            ax.scatter(selected_data[x_name], selected_data[y_name], alpha = 0.5, label=elemento)
            ax.set_xlabel(x_name)
            ax.set_ylabel(y_name)
    ax.legend(labels = ["Allow", "Drop"])
    ax.grid()

In [53]:
def analisiGrafica(data, cat_name):
    nomi_colonne = [col for col in data.columns if col != cat_name]
    
    n_vars = len(nomi_colonne)
    fig, axs = plt.subplots(nrows=n_vars, ncols=n_vars, figsize=(80, 80))
    
    for i, var1 in enumerate(nomi_colonne):
        for j, var2 in enumerate(nomi_colonne):
            disegnaGrafico(var1, var2, data, cat_name, axs[i, j])

    #plt.savefig("firewall.pdf")
    plt.show()

### Ora leggo i dati

In [54]:
data = pd.read_csv("log2.csv")
data.head(5)

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18


In [55]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Source Port           65532 non-null  int64 
 1   Destination Port      65532 non-null  int64 
 2   NAT Source Port       65532 non-null  int64 
 3   NAT Destination Port  65532 non-null  int64 
 4   Action                65532 non-null  object
 5   Bytes                 65532 non-null  int64 
 6   Bytes Sent            65532 non-null  int64 
 7   Bytes Received        65532 non-null  int64 
 8   Packets               65532 non-null  int64 
 9   Elapsed Time (sec)    65532 non-null  int64 
 10  pkts_sent             65532 non-null  int64 
 11  pkts_received         65532 non-null  int64 
dtypes: int64(11), object(1)
memory usage: 6.0+ MB


In [56]:
data.tail(5)

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
65527,63691,80,13237,80,allow,314,192,122,6,15,4,2
65528,50964,80,13485,80,allow,4680740,67312,4613428,4675,77,985,3690
65529,54871,445,0,0,drop,70,70,0,1,0,1,0
65530,54870,445,0,0,drop,70,70,0,1,0,1,0
65531,54867,445,0,0,drop,70,70,0,1,0,1,0


## Devo prepararmi i dati, action è string, mi serve in numero! 
Tipo Enum, uso labelEncoder

In [57]:
labelEncoder_action = LabelEncoder()
data["Action_numeric"] = labelEncoder_action.fit_transform(data['Action'])
data.head(5)

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received,Action_numeric
0,57222,53,54587,53,allow,177,94,83,2,30,1,1,0
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9,0
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1,0
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7,0
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18,0


In [58]:
data.columns

Index(['Source Port', 'Destination Port', 'NAT Source Port',
       'NAT Destination Port', 'Action', 'Bytes', 'Bytes Sent',
       'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent',
       'pkts_received', 'Action_numeric'],
      dtype='object')

In [59]:
feature = ['Source Port', 'Destination Port', 'NAT Source Port',
       'NAT Destination Port', 'Bytes Sent',
       'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent',
       'pkts_received']
output = ['Action_numeric']

selected_data = data[feature+output]
selected_data.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received,Action_numeric
0,57222,53,54587,53,94,83,2,30,1,1,0
1,56258,3389,56258,3389,1600,3168,19,17,10,9,0
2,6881,50321,43265,50321,118,120,2,1199,1,1,0
3,50553,3389,50553,3389,1438,1889,15,17,8,7,0
4,50002,443,45848,443,6778,18580,31,16,13,18,0


## Appunti sul dataset

* Nel seguente dataset non sono presenti valori null, certo ci sono 3 regole alla fine di bloccaggio con 0 in alcuni campi...
* Io i Byte li posso calcolare, è la somma di bytes sent e bytes received...

In [ ]:
analisiGrafica(selected_data, "Action_numeric")

## Creo il classificatore

In [ ]:
X = selected_data[feature].to_numpy()
y = selected_data[output].to_numpy()

In [ ]:
scaler = StandardScaler()
#Mi normalizza tutto!
X_norm = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, random_state=42, test_size=0.3)

In [ ]:
neural_network = MLPClassifier()

In [ ]:
neural_network.fit(X_train, y_train.reshape(-1))

In [ ]:
neural_network.score(X_test, y_test)